# Defense-RL — GRPO Fine-Tuning on Colab T4

This notebook:
1. Clones your Defense-RL code from HuggingFace
2. Installs dependencies
3. Runs GRPO fine-tuning on Qwen2.5-1.5B-Instruct
4. Saves LoRA adapters back to HuggingFace Hub

**Runtime**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────
!nvidia-smi

Sun Apr 26 03:06:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ── Cell 2: Clone your Space repo ─────────────────────────────────
!git clone https://huggingface.co/spaces/Bhaskar111/defense-ai defense-rl
%cd defense-rl
!ls

Cloning into 'defense-rl'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 149 (delta 54), reused 0 (delta 0), pack-reused 4 (from 1)
Receiving objects: 100% (149/149), 139.99 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/defense-rl
agent.py  defense_env	    models.py	    requirements.txt  train_colab.ipynb
agents	  Dockerfile	    openenv.yaml    server	      training
blog.md   grpo_results.png  pyproject.toml  start.sh	      train.py
data	  inference.py	    README.md	    tests	      train_space.py


In [ ]:
# ── Cell 3: Install dependencies (takes ~3 min) ───────────────────
!pip install -q \
    transformers>=4.45.0 \
    trl>=0.11.0 \
    peft>=0.12.0 \
    accelerate>=0.30.0 \
    datasets>=2.20.0 \
    bitsandbytes>=0.43.0 \
    huggingface_hub \
    torch --quiet

In [ ]:
# ── Cell 4: Set your HF token (for saving checkpoints) ────────────
import os
from google.colab import userdata

# Option A: Use Colab Secrets (recommended)
# Go to: Secrets (key icon on left) → Add → Name: HF_TOKEN, Value: hf_xxx
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Token loaded from Colab Secrets')
except:
    # Option B: Paste directly (less secure)
    HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # replace this
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Token set manually')

Token set manually


In [ ]:
# ── Cell 5: Dry run first (verify everything works) ───────────────
!python train.py \
    --backend rule \
    --episodes 30 \
    --dry-run

DEFENSE AI — GRPO MULTI-AGENT TRAINING
  Backend:    rule
  Model:      Qwen/Qwen2.5-1.5B-Instruct
  Episodes:   30
  Epochs:     3
  Dry run:    True
  Tasks:      ['task_easy', 'task_medium', 'task_hard']

[Setup] Initialising agents...

[Collect] Running 30 episodes across tasks: ['task_easy', 'task_medium', 'task_hard']
  [Ep 01/10 | task_easy] steps= 5 | reward=+0.797 | score=0.6575 | failed
  [Ep 02/10 | task_easy] steps= 5 | reward=+0.720 | score=0.6575 | failed
  [Ep 03/10 | task_easy] steps= 5 | reward=+1.275 | score=0.9900 | SOLVED
  [Ep 04/10 | task_easy] steps= 5 | reward=+0.797 | score=0.6575 | failed
  [Ep 05/10 | task_easy] steps= 3 | reward=+1.002 | score=0.9900 | SOLVED
  [Ep 06/10 | task_easy] steps= 3 | reward=+0.247 | score=0.4650 | failed
  [Ep 07/10 | task_easy] steps= 3 | reward=+1.002 | score=0.9900 | SOLVED
  [Ep 08/10 | task_easy] steps= 3 | reward=+0.325 | score=0.4650 | failed
  [Ep 09/10 | task_easy] steps= 3 | reward=+1.002 | score=0.9900 | SOLVED
  [Ep 10

In [ ]:
# ── Cell 6: FULL GRPO Training ────────────────────────────────────
# This fine-tunes Qwen2.5-1.5B-Instruct with GRPO on T4 GPU
# Expected time: ~30-60 min for 50 episodes, 3 epochs

!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 50 \
    --epochs 3 \
    --batch-size 1 \
    --lora-rank 16 \
    --device cuda \
    --checkpoint-dir checkpoints

DEFENSE AI — GRPO MULTI-AGENT TRAINING
  Backend:    local
  Model:      Qwen/Qwen2.5-1.5B-Instruct
  Episodes:   50
  Epochs:     3
  Dry run:    False
  Tasks:      ['task_easy', 'task_medium', 'task_hard']

[Setup] Initialising agents...
[RadarAgent] Loading Qwen/Qwen2.5-1.5B-Instruct locally...
config.json: 100% 660/660 [00:00<00:00, 3.88MB/s]
tokenizer_config.json: 7.30kB [00:00, 5.00MB/s]
vocab.json: 2.78MB [00:00, 74.9MB/s]
merges.txt: 1.67MB [00:00, 124MB/s]
tokenizer.json: 7.03MB [00:00, 159MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.09G/3.09G [00:28<00:00, 108MB/s]
Loading weights: 100% 338/338 [00:06<00:00, 54.15it/s, Materializing param=model.norm.weight] 
generation_config.json: 100% 242/242 [00:00<00:00, 1.09MB/s]
Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR

In [ ]:
# ── Cell 7: Save trained adapters to HuggingFace Hub ─────────────
from huggingface_hub import HfApi, create_repo
import os

api = HfApi(token=os.environ['HF_TOKEN'])
USERNAME = 'Bhaskar111'  # your HF username

# Create model repos and upload adapters
for agent in ['radar', 'actor']:
    repo_id = f'{USERNAME}/defense-rl-{agent}-adapter'
    checkpoint_path = f'checkpoints/{agent}'

    if os.path.exists(checkpoint_path):
        create_repo(repo_id, repo_type='model', exist_ok=True, token=os.environ['HF_TOKEN'])
        api.upload_folder(
            folder_path=checkpoint_path,
            repo_id=repo_id,
            repo_type='model',
            token=os.environ['HF_TOKEN']
        )
        print(f'Uploaded {agent} adapter → https://huggingface.co/{repo_id}')
    else:
        print(f'No checkpoint found at {checkpoint_path}')

In [ ]:
# ── Cell 8: Quick evaluation after training ───────────────────────
!python train.py \
    --backend local \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --episodes 30 \
    --epochs 1 \
    --dry-run \
    --resume checkpoints